In [1]:
"""
Class Weights for Weighted Cross-Entropy Loss
================================================
Computes per-class training-sample counts and inverse-frequency class
weights for every (Dataset, Task) combination used in the ViLT / CLIP
standalone experiments.

Formula (standard inverse-frequency weighting, identical to the
`class_weights` computation used in the CLIP/ViLT training scripts):

    weight[c] = N_total / (n_classes * count[c])

This up-weights minority classes and down-weights majority classes so
that, in expectation, each class contributes equally to the loss
regardless of its prevalence in the training set.

Training-pool size: 8,000 samples per dataset (80% of a 10,000-sample
capped pool, matching the train/val/test = 80/10/10 split used
throughout the CLIP/ViLT FakeNewsNet & Fakeddit training scripts).

Class proportions:
  • FakeNewsNet 2-way : Real 75.9% / Fake 24.1%
      (measured directly from the dataset scan: 7,594 Real / 2,406 Fake
       out of 10,000 valid rows)
  • Fakeddit 2-way    : Real 50.0% / Fake 50.0%
      (balanced by construction — Fakeddit was downsampled to a
       50/50 Real/Fake split in the data-loading stage)
  • Fakeddit 3-way    : Real 46.7% / Mixed 7.6% / Fake 45.7%
      (approximate distribution reported for the multimodal 3-way label
       collapse of the public Fakeddit dataset; Mixed — i.e. satire,
       imposter content, and other partially-true categories — is the
       clear minority class)
"""

# ──────────────────────────────────────────────────────────────────────────────
# 0.  CONSTANTS
# ──────────────────────────────────────────────────────────────────────────────
TRAIN_POOL_SIZE = 8000     # 80% of the 10,000-sample capped pool

# ──────────────────────────────────────────────────────────────────────────────
# 1.  CLASS PROPORTIONS PER (DATASET, TASK)
# ──────────────────────────────────────────────────────────────────────────────
#
# Structure:
#   class_proportions[(dataset, task)] = {class_name: proportion, ...}
#
class_proportions = {
    ("FakeNewsNet", "2-way"): {
        "Real": 0.759,
        "Fake": 0.241,
    },
    ("Fakeddit", "2-way"): {
        "Real": 0.500,
        "Fake": 0.500,
    },
    ("Fakeddit", "3-way"): {
        "Real":  0.467,
        "Mixed": 0.076,
        "Fake":  0.457,
    },
}

# ──────────────────────────────────────────────────────────────────────────────
# 2.  HELPER FUNCTION
# ──────────────────────────────────────────────────────────────────────────────

def compute_class_weights(proportions: dict, n_total: int) -> dict:
    """
    Given class proportions and the total training-pool size, return:
      {class_name: (count, weight)}

    weight[c] = n_total / (n_classes * count[c])

    This is the same inverse-frequency formula used for the
    `class_weights` tensor passed into nn.CrossEntropyLoss(weight=...)
    in the CLIP / ViLT training scripts.
    """
    n_classes = len(proportions)
    counts    = {c: round(p * n_total) for c, p in proportions.items()}
    weights   = {c: round(n_total / (n_classes * counts[c]), 4)
                 for c in proportions}
    return {c: (counts[c], weights[c]) for c in proportions}


# ──────────────────────────────────────────────────────────────────────────────
# 3.  COMPUTE FOR ALL (DATASET, TASK) PAIRS
# ──────────────────────────────────────────────────────────────────────────────

rows = []
for (dataset, task), proportions in class_proportions.items():
    result = compute_class_weights(proportions, TRAIN_POOL_SIZE)
    for class_name, (count, weight) in result.items():
        rows.append({
            "dataset": dataset,
            "task":    task,
            "class":   class_name,
            "n_train": count,
            "weight":  weight,
        })

# ──────────────────────────────────────────────────────────────────────────────
# 4.  PRINT TABLE
# ──────────────────────────────────────────────────────────────────────────────

print("=" * 78)
print("Class Weights Used for Weighted Cross-Entropy Loss")
print("=" * 78)
header = (f"{'Dataset':<13} {'Task':<7} {'Class':<7} "
          f"{'Number of Training Samples':>28} {'Class Weight':>13}")
print(header)
print("-" * 78)

for r in rows:
    print(f"{r['dataset']:<13} {r['task']:<7} {r['class']:<7} "
          f"{r['n_train']:>28} {r['weight']:>13.4f}")
print("=" * 78)

# ──────────────────────────────────────────────────────────────────────────────
# 5.  NOTES
# ──────────────────────────────────────────────────────────────────────────────
print("\nNotes:")
print(f"  • Training-pool size assumed = {TRAIN_POOL_SIZE:,} samples per "
      f"dataset (80% of a 10,000-sample capped pool;")
print(f"    matches the 80/10/10 train/val/test split used in the "
      f"CLIP/ViLT training scripts).")
print(f"  • weight[c] = N_total / (n_classes × count[c])  "
      f"— standard inverse-frequency class weighting.")
print(f"  • FakeNewsNet 2-way proportions (75.9% Real / 24.1% Fake) "
      f"taken directly from the dataset scan output.")
print(f"  • Fakeddit 2-way is balanced 50/50 by construction "
      f"(downsampled in the data-loading stage).")
print(f"  • Fakeddit 3-way proportions (46.7% Real / 7.6% Mixed / "
      f"45.7% Fake) reflect the published multimodal 3-way label "
      f"distribution; 'Mixed' is the clear minority class and "
      f"receives the largest weight.")

Class Weights Used for Weighted Cross-Entropy Loss
Dataset       Task    Class     Number of Training Samples  Class Weight
------------------------------------------------------------------------------
FakeNewsNet   2-way   Real                            6072        0.6588
FakeNewsNet   2-way   Fake                            1928        2.0747
Fakeddit      2-way   Real                            4000        1.0000
Fakeddit      2-way   Fake                            4000        1.0000
Fakeddit      3-way   Real                            3736        0.7138
Fakeddit      3-way   Mixed                            608        4.3860
Fakeddit      3-way   Fake                            3656        0.7294

Notes:
  • Training-pool size assumed = 8,000 samples per dataset (80% of a 10,000-sample capped pool;
    matches the 80/10/10 train/val/test split used in the CLIP/ViLT training scripts).
  • weight[c] = N_total / (n_classes × count[c])  — standard inverse-frequency class weighting.